# Energy-Based Models (EBMs)
## Learning Unnormalised Distributions with Contrastive Divergence

**MLNN Tutorial · University of Hertfordshire · 2025**  
GitHub: https://github.com/revtheundefined/ebm-tutorial

### Resources used
- LeCun, Y. et al. (2006) 'A Tutorial on Energy-Based Learning'. https://yann.lecun.com/exdb/publis/pdf/lecun-06.pdf
- Hinton, G.E. (2002) 'Training products of experts by minimizing contrastive divergence', *Neural Computation*, 14(8). https://www.cs.toronto.edu/~hinton/absps/tr00-004.pdf
- Hyvarinen, A. (2005) 'Estimation of Non-Normalized Statistical Models by Score Matching', *JMLR*. https://jmlr.org/papers/v6/hyvarinen05a.html
- Du, Y. and Mordatch, I. (2019) 'Implicit Generation and Modeling with Energy Based Models'. https://arxiv.org/abs/1903.08689
- Song, Y. and Ermon, S. (2019) 'Generative Modeling by Estimating Gradients of the Data Distribution'. https://arxiv.org/abs/1907.05600
- Grathwohl, W. et al. (2020) 'Your Classifier is Secretly an Energy Based Model'. https://arxiv.org/abs/1912.03263

### Accessibility
All figures use colourblind-safe crimson/teal palettes. Heatmaps use RdYlGn colourmap. Alt-text in every Markdown cell.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                       'torch', 'matplotlib', 'numpy', 'scipy', '--quiet'])

In [ ]:
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.optim as optim
from scipy.ndimage import uniform_filter1d
import warnings; warnings.filterwarnings('ignore')

BG='#0E0808'; CARD='#1A0F0F'; GRID='#2A1818'
CRIM='#F87171'; TEAL='#2DD4BF'; GOLD='#FCD34D'
LIME='#86EFAC'; ORNG='#FB923C'; TEXT='#FEF2F2'; MUTE='#FDA4AF'

plt.rcParams.update({'figure.facecolor':BG,'axes.facecolor':CARD,
                     'font.family':'monospace','font.size':10})
def sa(ax):
    ax.set_facecolor(CARD); ax.tick_params(colors=MUTE,labelsize=9)
    ax.xaxis.label.set_color(TEXT); ax.yaxis.label.set_color(TEXT); ax.title.set_color(TEXT)
    for sp in ax.spines.values(): sp.set_edgecolor(GRID)
    ax.grid(True,color=GRID,linewidth=0.7,alpha=0.8)

torch.manual_seed(42); np.random.seed(42)
print('Setup complete.')

---
## Section 1: The Energy-Based Model Framework

An EBM defines a probability distribution as:

p_θ(x) = exp(-E_θ(x)) / Z_θ

where E_θ(x) is a learned energy function and Z_θ = ∫exp(-E_θ(x))dx is the intractable partition function.

**Key insight**: We never need to compute Z_θ for training — only for evaluation of absolute likelihoods.

**Alt-text:** Code verifying the energy-to-probability conversion and demonstrating why Z is intractable.

In [ ]:
# ── EBM framework: energy → probability ──────────────────────────────────────
# 1D example: bimodal distribution
x_grid = np.linspace(-4, 4, 1000)
dx     = x_grid[1] - x_grid[0]

def energy(x, mu1=1.5, mu2=-1.2, sigma1=0.4, sigma2=0.5):
    """Bimodal energy function — two Gaussian wells."""
    return -(2.0 * np.exp(-((x-mu1)**2)/(2*sigma1**2)) +
             1.5 * np.exp(-((x-mu2)**2)/(2*sigma2**2)))

E = energy(x_grid)
unnorm_prob = np.exp(-E)         # exp(-E(x)) — unnormalised
Z = unnorm_prob.sum() * dx       # partition function (numerically)
norm_prob = unnorm_prob / Z      # normalised p(x)

print(f'Partition function Z = {Z:.4f}')
print(f'Probability integrates to: {norm_prob.sum()*dx:.6f} (should be 1.0)')
print()
print('Why Z is intractable in high dimensions:')
print(f'  1D grid: {len(x_grid)} points')
print(f'  2D grid: {len(x_grid)**2:,} points')
print(f'  10D grid: {len(x_grid)**10:.2e} points — impossible!')
print()
print('Solution: avoid computing Z by using contrastive divergence or score matching.')

---
## Section 2: Contrastive Divergence Training

The MLE gradient for EBMs is:

∇_θ log p_θ(x) = -∇_θ E_θ(x) + E_{p_θ}[∇_θ E_θ(x̃)]

The first term is cheap (just evaluate E at data). The second term requires samples from p_θ — expensive MCMC. CD-k approximates this with k MCMC steps starting from data.

**Alt-text:** Full EBM implementation with CD-1 training and Langevin MCMC sampling.

In [ ]:
class EnergyNet(nn.Module):
    """
    Neural energy function E_θ(x): R^d -> R.
    Low output = high probability region.
    Uses SiLU activation (smooth, avoids dead neurons).
    """
    def __init__(self, dim=2, hidden=128, n_layers=3):
        super().__init__()
        layers = [nn.Linear(dim, hidden), nn.SiLU()]
        for _ in range(n_layers - 1):
            layers += [nn.Linear(hidden, hidden), nn.SiLU()]
        layers.append(nn.Linear(hidden, 1))
        self.net = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.net(x).squeeze(-1)


def langevin_mcmc(energy_fn, n_samples, dim=2, steps=20, step_size=0.1, noise=0.2, init=None):
    """
    Langevin MCMC sampling from EBM.
    Update rule: x_{t+1} = x_t - (η/2)∇_x E(x_t) + √η · ε,  ε ~ N(0,I)
    
    The gradient term pushes x toward low-energy regions.
    The noise term ensures exploration and correct stationary distribution.
    """
    energy_fn.eval()
    x = torch.randn(n_samples, dim) if init is None else init.clone()
    
    for _ in range(steps):
        x = x.detach().requires_grad_(True)
        with torch.enable_grad():
            e = energy_fn(x)
            grad = torch.autograd.grad(e.sum(), x)[0]
        with torch.no_grad():
            x = x - (step_size / 2) * grad + np.sqrt(step_size) * noise * torch.randn_like(x)
    
    energy_fn.train()
    return x.detach()


def train_ebm(model, data, epochs=300, lr=1e-3, cd_steps=20, step_size=0.1):
    """
    Train EBM with Contrastive Divergence.
    
    CD gradient: ∇_θ E(x_neg) - ∇_θ E(x_pos)
    - x_pos = real data (push energy DOWN)
    - x_neg = MCMC samples from model (push energy UP)
    
    Regularisation: L2 penalty on energy values to prevent collapse.
    """
    opt = optim.Adam(model.parameters(), lr=lr)
    losses = []
    
    for ep in range(epochs):
        idx   = torch.randint(0, len(data), (64,))
        x_pos = data[idx]    # positive samples (real data)
        
        # Negative samples via Langevin MCMC
        x_neg = langevin_mcmc(model, n_samples=64, dim=data.shape[1],
                               steps=cd_steps, step_size=step_size)
        
        # CD loss: E(x_neg) - E(x_pos)
        # Minimising this pushes E(x_pos) down and E(x_neg) up
        e_pos = model(x_pos)
        e_neg = model(x_neg)
        loss  = e_pos.mean() - e_neg.mean()
        # L2 regularisation on energy magnitudes
        loss  = loss + 0.01 * (e_pos**2 + e_neg**2).mean()
        
        opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        losses.append(loss.item())
        
        if (ep+1) % 100 == 0:
            print(f'Epoch {ep+1}/{epochs}: CD loss = {loss.item():.4f}')
    
    return losses

# Generate 2D mixture data
def make_mixture(n=800):
    centers = [(1.5,0),(-1.5,0),(0,1.5),(0,-1.5)]
    data = []
    for _ in range(n):
        c = centers[np.random.randint(4)]
        data.append([c[0]+np.random.randn()*0.25, c[1]+np.random.randn()*0.25])
    return torch.tensor(data, dtype=torch.float32)

torch.manual_seed(42)
data_2d = make_mixture(800)
ebm = EnergyNet(dim=2, hidden=64, n_layers=3)
print('Training EBM on 4-mode Gaussian mixture...')
losses = train_ebm(ebm, data_2d, epochs=300, lr=1e-3)
print('Training complete.')

---
## Section 3: Score Matching — Avoiding Z Entirely

Score matching (Hyvärinen, 2005) avoids Z by directly matching the score function ∇_x log p(x) instead of the density. The score matching loss is:

J(θ) = E_p[||s_θ(x) - ∇_x log p_data(x)||²]
      = E_p[tr(∇_x s_θ(x)) + ½||s_θ(x)||²] + const

This is computable without knowing Z or sampling from the model!

**Alt-text:** Score matching implementation demonstrating how to learn the gradient of log p without computing the normalisation constant.

In [ ]:
# ── Score matching: learn ∇_x log p without computing Z ──────────────────────
class ScoreNetwork(nn.Module):
    """
    Score network: learns s_θ(x) ≈ ∇_x log p(x) = -∇_x E(x).
    Output dim = input dim (vector field).
    """
    def __init__(self, dim=2, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, dim)  # output: score vector
        )
    def forward(self, x): return self.net(x)

def score_matching_loss(score_net, x):
    """
    Explicit score matching loss (Hyvärinen, 2005):
    J = E[tr(∇_x s_θ(x)) + ½||s_θ(x)||²]
    
    tr(∇_x s_θ) = sum of diagonal of Jacobian of s_θ w.r.t. x
    """
    x = x.requires_grad_(True)
    s = score_net(x)  # (N, d)
    
    # Compute trace of Jacobian via autograd
    jacobian_trace = torch.zeros(x.shape[0])
    for i in range(x.shape[1]):
        grad = torch.autograd.grad(s[:, i].sum(), x, create_graph=True)[0]
        jacobian_trace += grad[:, i]
    
    # Score matching loss
    loss = jacobian_trace + 0.5 * (s**2).sum(dim=1)
    return loss.mean()

# Train score network on 1D mixture
score_net = ScoreNetwork(dim=1, hidden=32)
opt_s = optim.Adam(score_net.parameters(), lr=1e-3)

# 1D bimodal data
data_1d = torch.cat([
    torch.randn(400, 1) * 0.4 + 1.5,
    torch.randn(400, 1) * 0.5 - 1.2
])

sm_losses = []
for ep in range(500):
    idx  = torch.randint(0, len(data_1d), (128,))
    loss = score_matching_loss(score_net, data_1d[idx])
    opt_s.zero_grad(); loss.backward(); opt_s.step()
    sm_losses.append(loss.item())

# Evaluate: compare learned score to true score
x_eval = torch.linspace(-4, 4, 100).reshape(-1, 1)
x_np   = x_eval.numpy().ravel()

# True score = d/dx log p(x) = -d/dx E(x)
log_p_unnorm = (0.6 * np.exp(-((x_np-1.5)**2)/(2*0.16)) +
                0.4 * np.exp(-((x_np+1.2)**2)/(2*0.25)))
true_score = np.gradient(np.log(log_p_unnorm + 1e-10), x_np)

with torch.no_grad():
    learned_score = score_net(x_eval).numpy().ravel()

# Normalise for comparison
scale = np.abs(true_score).max() / (np.abs(learned_score).max() + 1e-8)
print(f'Score matching converged. Final loss: {sm_losses[-1]:.4f}')
print(f'Correlation between true and learned score: '
      f'{np.corrcoef(true_score, learned_score*scale)[0,1]:.4f}')
print('High correlation confirms score matching recovers gradient of log p.')

---
## Summary Table

| Model | Exact p(x)? | Z tractable? | Generates? | Training method |
|-------|-------------|--------------|------------|------------------|
| EBM (CD) | No | No | MCMC | Contrastive Divergence |
| VAE | No (ELBO) | Yes* | Yes | ELBO |
| GAN | No | No | Yes | Adversarial |
| Flow | **Yes** | **Yes** | Yes | Max likelihood |
| Diffusion | No | No | Yes | Denoising score matching |

## References

1. LeCun, Y. et al. (2006) 'A Tutorial on Energy-Based Learning'. https://yann.lecun.com/exdb/publis/pdf/lecun-06.pdf
2. Hinton, G.E. (2002) 'Training products of experts by minimizing contrastive divergence', *Neural Computation*, 14(8). https://www.cs.toronto.edu/~hinton/absps/tr00-004.pdf
3. Hyvarinen, A. (2005) 'Estimation of Non-Normalized Statistical Models by Score Matching', *JMLR*. https://jmlr.org/papers/v6/hyvarinen05a.html
4. Du, Y. and Mordatch, I. (2019) 'Implicit Generation and Modeling with Energy Based Models'. https://arxiv.org/abs/1903.08689
5. Song, Y. and Ermon, S. (2019) 'Generative Modeling by Estimating Gradients of the Data Distribution'. https://arxiv.org/abs/1907.05600
6. Grathwohl, W. et al. (2020) 'Your Classifier is Secretly an Energy Based Model'. https://arxiv.org/abs/1912.03263